# Tutorial 01: Quickstart

The shortest path from a synthetic detector frame to an extracted spectrum, in
about a minute of runtime.

**What you get:** a 2-D fiber-flat image, a trace model of every fiber, and a
row-stacked spectrum.

**What this tutorial skips:** the internals. The trace model, the extraction
methods and the wavelength calibration each get their own tutorial, numbered by
pipeline order:

| Tutorial | Subject |
| --- | --- |
| 01 | Quickstart (this one) |
| 02 | Image preprocessing |
| 03 | Fiber identification and tracing |
| 04 | Spectral extraction |
| 05 | Wavelength calibration |
| 06 | Logging |
| 07 | End to end |

**Before you start:** install the package and Jupyter in an environment that
has `numpy`, `astropy`, `matplotlib` and `scipy`; see `docs/CONTRIBUTING.md`.

## 1. Imports

In [ ]:
# Uncomment for interactive pan and zoom (requires `pip install ipympl`).
# %matplotlib ipympl

import numpy as np
import matplotlib.pyplot as plt

from amasedrp.simulate import synthetic_fiber_flat, synthetic_arc
from amasedrp.reduction import identify_and_trace_fibers, extract_spectra

## 2. A synthetic fiber flat

A fiber flat is an exposure of a continuum lamp through the fibers. It shows
where every fiber sits, so the pipeline uses it to find the fibers and to model
their traces.

`amasedrp.simulate` builds one whose geometry matches the AMASE-P prototype:
fibers 8 px apart with a 2 px Gaussian width, grouped into blocks that are
26 px apart. The fibers are straight, so nothing here needs the curvature that
a real flat has.

In [ ]:
flat, true_centers = synthetic_fiber_flat(
    n_rows=1200,
    n_blocks=3,
    n_fibers_per_block=5,
)

print(f"flat shape      : {flat.shape}")
print(f"fibers expected : {len(true_centers)}")
print(f"first 5 centers : {[round(c, 1) for c in true_centers[:5]]}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(flat, aspect="auto", origin="lower", cmap="gray")
ax.set_xlabel("cross-dispersion [pixel]")
ax.set_ylabel("dispersion [row]")
ax.set_title("Synthetic fiber flat: 3 blocks of 5 fibers")
plt.show()

## 3. Identify and trace the fibers

`identify_and_trace_fibers` does two things. It finds each fiber (the
`FiberMap`), then it follows each fiber from row to row and fits a polynomial
through the positions (the `TraceMask`).

The two numbers below are the ones to check. A mismatch means the frame is not
what the pipeline expected, or the fiber configuration in the header is wrong.

In [ ]:
fibermap, tracemask = identify_and_trace_fibers(
    image=flat,
    n_blocks_expected=3,
    n_fibers_per_block_expected=5,
    poly_deg=3,
)

print(f"fibers found : {fibermap.n_fibers}   (expected {len(true_centers)})")
print(f"blocks found : {len(np.unique(fibermap['BLOCKID']))}   (expected 3)")
print(f"valid fibers : {int(fibermap['VALID'].sum())}")
print(f"trace rows   : {tracemask.coeffs.shape[1] - 1} polynomial degree")
print()
print(fibermap[:5])

### Check the trace against the truth

The synthetic fibers are straight, so the trace at any row must return the
center of the fiber. This is the smallest check that fails when the tracing
breaks.

In [ ]:
rows = np.arange(flat.shape[0])
positions = tracemask.eval(rows)
residual = positions - np.asarray(true_centers)[:, np.newaxis]

print(f"trace residual : rms {np.sqrt(np.mean(residual ** 2)):.4f} px, "
      f"max {np.max(np.abs(residual)):.4f} px")

fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(flat, aspect="auto", origin="lower", cmap="gray")
ax.plot(positions.T, rows, "r-", lw=0.6, alpha=0.7)
ax.set_xlabel("cross-dispersion [pixel]")
ax.set_ylabel("dispersion [row]")
ax.set_title("Fiber traces over the flat")
plt.show()

## 4. Extract the spectra

The science frame has to be extracted with the trace model. Here a synthetic
arc stands in for the science frame: a lamp with three emission lines, taken
through the same fibers, so it uses the same trace model.

`method="boxcar"` sums the pixels inside an aperture around each trace. It is
the fast, robust choice, and the one to reach for first.

In [ ]:
science, wavelength = synthetic_arc(
    flat,
    lines=np.array([4657.9012, 4764.8646, 4965.0795]),
    wavelength_zero=4600.0,
    dispersion=0.35,
    line_fwhm=1.2,
    noise_std=0.01,
)

frame = extract_spectra(
    image=science,
    tracemask=tracemask,
    fibermap=fibermap,
    method="boxcar",
    aperture_radius=3,
)

print(f"frame shape  : {frame.shape}")
print(f"fibers       : {frame.n_fibers}")
print(f"wavelength bins: {frame.n_wave}")
print(f"flux range   : [{np.nanmin(frame.flux):.2f}, {np.nanmax(frame.flux):.2f}]")

The `wave` array of the frame is still in pixel units, because no wavelength
solution has been applied yet. Tutorial 05 solves that mapping from an arc
exposure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].imshow(science, aspect="auto", origin="lower", cmap="gray")
axes[0].set_xlabel("cross-dispersion [pixel]")
axes[0].set_ylabel("dispersion [row]")
axes[0].set_title("Synthetic arc frame")

for i in range(frame.n_fibers):
    axes[1].plot(frame.wave, frame.flux[i], lw=0.7, alpha=0.8)
axes[1].set_xlabel("pixel (dispersion direction)")
axes[1].set_ylabel("flux [ADU]")
axes[1].set_title("Extracted spectra, boxcar")
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## Summary

| Step | Call | Result |
| --- | --- | --- |
| Build a flat | `synthetic_fiber_flat()` | 2-D image, true fiber centers |
| Find and trace | `identify_and_trace_fibers(image, ...)` | `FiberMap`, `TraceMask` |
| Extract | `extract_spectra(image, tracemask, fibermap, ...)` | `FiberFrame` |

On this frame the pipeline finds 15 fibers in 3 blocks, and the trace
reproduces the true fiber centers to an rms of 0.07 px.

**Where to go next**

- Tutorial 03 explains block detection, the per-fiber fit domain, and what the
  trace residual does when a fiber runs off the frame.
- Tutorial 04 compares boxcar against optimal extraction.
- Tutorial 05 turns rows into wavelengths and measures the line-spread
  function.